# 🩺 Verificação das Dimensões das Imagens do MedMNIST

Este notebook verifica as **dimensões, canais de cor, formatos de tensores e metadados** das imagens do MedMNIST baixadas localmente.

Ele suporta dois formatos comuns:
1. **Via cache local do Hugging Face / datasets** (padrão utilizado neste projeto, ex: `danjacobellis/pathmnist_224`)
2. **Via arquivo local `.npz`** (formato oficial de download do MedMNIST, ex: `pathmnist.npz`)

In [ ]:
import os
import numpy as np
import torch
import torchvision.transforms as T
import matplotlib.pyplot as plt
from PIL import Image

# Força o modo offline para utilizar apenas os arquivos em cache local
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"

print("✅ Bibliotecas importadas com sucesso!")

## 1. Verificação via Cache Local Hugging Face (`datasets`)
Verifica o dataset baixado localmente no cache da máquina (`~/.cache/huggingface/datasets`).

In [ ]:
from datasets import load_dataset

# Nome do dataset MedMNIST (substitua por 'breastmnist_224', 'octmnist_224', etc., se desejar)
dataset_name = "danjacobellis/pathmnist_224"

try:
    print(f"Carregando '{dataset_name}' do cache local...")
    ds = load_dataset(dataset_name)
    print(f"✅ Dataset carregado com sucesso!")
    print(f"Divisões (Splits): {list(ds.keys())}")
    
    for split in ds.keys():
        print(f"  - {split}: {len(ds[split])} imagens")
    
    # Pega uma amostra do conjunto de treino
    sample = ds["train"][0]
    img_pil = sample["image"]
    label = sample["label"]
    
    print("\n" + "="*40)
    print("DIMENSÕES DA IMAGEM")
    print("="*40)
    print(f"1. Formato PIL Image:")
    print(f"   - Tipo: {type(img_pil)}")
    print(f"   - Modo: {img_pil.mode} ({'RGB - 3 Canais' if img_pil.mode == 'RGB' else 'Escala de Cinza - 1 Canal'})")
    print(f"   - Tamanho (Largura x Altura): {img_pil.size} pixels (W={img_pil.width}, H={img_pil.height})")
    
    # 2. Conversão para NumPy
    img_np = np.array(img_pil)
    print(f"\n2. Formato NumPy Array:")
    print(f"   - Shape (H, W, C): {img_np.shape}")
    print(f"   - Altura: {img_np.shape[0]} px | Largura: {img_np.shape[1]} px | Canais: {img_np.shape[2] if img_np.ndim == 3 else 1}")
    print(f"   - Tipo de dados: {img_np.dtype} (Valores de {img_np.min()} a {img_np.max()})")
    
    # 3. Conversão para PyTorch Tensor (ToTensor normaliza para [0.0, 1.0])
    transform = T.ToTensor()
    img_tensor = transform(img_pil)
    print(f"\n3. Formato PyTorch Tensor:")
    print(f"   - Shape (C, H, W): {tuple(img_tensor.shape)}")
    print(f"   - Canais: {img_tensor.shape[0]}")
    print(f"   - Altura: {img_tensor.shape[1]}")
    print(f"   - Largura: {img_tensor.shape[2]}")
    print(f"   - Dtype: {img_tensor.dtype} (Intervalo: [{img_tensor.min():.2f}, {img_tensor.max():.2f}])")
    print(f"   - Rótulo (Label): {label}")

except Exception as e:
    print(f"❌ Erro ao carregar dataset: {e}")

## 2. Verificação via Arquivo Local `.npz` (Download Oficial MedMNIST)
Se você baixou diretamente o arquivo `.npz` do MedMNIST (ex: `pathmnist.npz`), esta função inspeciona todas as matrizes salvas.

In [ ]:
def inspect_medmnist_npz(npz_file_path: str):
    """Carrega um arquivo .npz do MedMNIST e imprime dimensões detalhadas de cada split."""
    if not os.path.exists(npz_file_path):
        print(f"⚠️ Arquivo '{npz_file_path}' não encontrado no caminho indicado.")
        print("Dica: Ajuste a variável 'npz_file_path' com o caminho completo do seu arquivo .npz.")
        return
    
    data = np.load(npz_file_path)
    print(f"✅ Arquivo aberto: {npz_file_path}")
    print(f"Chaves presentes no .npz: {list(data.keys())}\n")
    
    for split in ["train", "val", "test"]:
        img_key = f"{split}_images"
        lbl_key = f"{split}_labels"
        
        if img_key in data:
            images = data[img_key]
            labels = data[lbl_key] if lbl_key in data else None
            
            num_imgs = images.shape[0]
            height = images.shape[1]
            width = images.shape[2]
            channels = images.shape[3] if images.ndim == 4 else 1
            
            print(f"🔹 Split: {split.upper()}")
            print(f"   - Shape das imagens: {images.shape}")
            print(f"   - Quantidade de imagens: {num_imgs:,}")
            print(f"   - Resolução: {width} x {height} pixels")
            print(f"   - Canais: {channels} ({'RGB' if channels == 3 else 'Grayscale'})")
            print(f"   - Tipo de dados: {images.dtype}")
            if labels is not None:
                print(f"   - Shape dos rótulos: {labels.shape}")
            print()

# Exemplo de teste: altere para o caminho do seu arquivo se tiver o .npz
inspect_medmnist_npz("pathmnist.npz")

## 3. Visualização de Amostras e Dimensões
Exibe a imagem carregada com suas dimensões e a distribuição de valores dos pixels.

In [ ]:
if 'img_np' in locals():
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    
    # Amostra visual
    if img_np.ndim == 2 or (img_np.ndim == 3 and img_np.shape[2] == 1):
        axes[0].imshow(img_np.squeeze(), cmap="gray")
    else:
        axes[0].imshow(img_np)
    axes[0].set_title(f"Amostra | Shape: {img_np.shape}")
    axes[0].axis("off")
    
    # Histograma dos valores de pixel
    axes[1].hist(img_np.ravel(), bins=50, color="royalblue", alpha=0.75, edgecolor="black")
    axes[1].set_title("Distribuição dos Valores de Pixels (0-255)")
    axes[1].set_xlabel("Valor do Pixel")
    axes[1].set_ylabel("Contagem")
    axes[1].grid(axis="y", linestyle="--", alpha=0.6)
    
    plt.tight_layout()
    plt.show()
else:
    print("Nenhuma imagem carregada na seção 1.")

## 📌 Resumo das Convenções de Dimensão

| Biblioteca / Estrutura | Convenção de Dimensão | Exemplo PathMNIST (224x224) | Exemplo MedMNIST Padrão (28x28) |
|---|---|---|---|
| **PIL Image** | `(Width, Height)` | `(224, 224)` | `(28, 28)` |
| **NumPy Array** | `(Height, Width, Channels)` | `(224, 224, 3)` | `(28, 28, 3)` ou `(28, 28)` |
| **PyTorch Tensor** | `(Channels, Height, Width)` | `torch.Size([3, 224, 224])` | `torch.Size([3, 28, 28])` |
| **DataLoader Batch** | `(Batch, Channels, Height, Width)` | `torch.Size([B, 3, 224, 224])` | `torch.Size([B, 3, 28, 28])` |